# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Computational Physics & Mathematical Integrators

---
*Note: This notebook implements a kinematic engine for particle scattering under Coulomb potential fields. Avoiding unstable methods like Euler, it rigorously uses a Symplectic Verlet Integrator to process spatial trajectories, guaranteeing absolute energy conservation of the Hamiltonian system. The product of this intensive numerical exercise is the empirical statistical deduction of a physical cross-section.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from numba import jit
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

# Physical constants naturalized to the simulated environment
k_e = 1.0  # Scaled Coulomb constant
Q_au = 79.0 # Gold Charge (Target nucleus)

# Time discretization
DT = 0.05
STEPS = 5000

### 1. Ingestion of the Monte Carlo Ensemble
The set of initial conditions (orthogonal positions and moments) for 5,000 pre-generated alpha particles is loaded.

In [ ]:
df_ensemble = pd.read_csv('../data/rutherford_initial_particles.csv')

# We extract numpy arrays to speed up Numba (Just-In-Time Compilation)
x_init = df_ensemble['x'].values
y_init = df_ensemble['y'].values
vx_init = df_ensemble['vx'].values
vy_init = df_ensemble['vy'].values
q1 = df_ensemble['charge_q1'].values[0]
m1 = df_ensemble['mass_m1'].values[0]

print(f"[*] Ensemble retrieved: {len(x_init)} alpha particles (Charge +{q1}, Mass {m1}u). Asymmetric 2D motion.")

### 2. Velocity-Verlet Symplectic Integrator (Vectorial Differential Calculus)
The algorithm computes central forces at each time difference $(dt)$, updates positions, recalculates gradients, and updates velocities. It is compiled with Numba (`@jit`) to replicate the multi-threaded speed (in silico) that C++ or C# would offer natively compared to Python's standard GIL cycle.

In [ ]:
@jit(nopython=True)
def compute_coulomb_force(x, y, q1_val, m1_val):
    """
    Vectorized computation of the inverse square of distance.
    F = (k_e * q1 * Q_au) / r^2 in radial direction.
    Returns accelerations ax, ay.
    """
    r_sq = x**2 + y**2
    # We prevent purely central collision divergences by adding a smoothing factor
    r_sq = np.where(r_sq < 1.0, 1.0, r_sq)
    r = np.sqrt(r_sq)
    
    force_mag = (k_e * q1_val * Q_au) / r_sq
    
    # Decomposition of the (repulsive) radial force vector
    ax = (force_mag / m1_val) * (x / r)
    ay = (force_mag / m1_val) * (y / r)
    
    return ax, ay

@jit(nopython=True)
def verlet_integrator(x, y, vx, vy, dt, steps, q1_val, m1_val):
    """
    Evolves the dynamic Hamiltonian system.
    We use Velocity-Verlet to not artificially drain thermal energy.
    """
    N = len(x)
    
    # Arrays to store trajectories of a subset (for trace plots)
    n_traces = 50
    x_trace = np.zeros((steps, n_traces))
    y_trace = np.zeros((steps, n_traces))
    
    # Initial acceleration conditions
    ax, ay = compute_coulomb_force(x, y, q1_val, m1_val)
    
    for t in range(steps):
        # 1. Update Position: r(t+dt) = r(t) + v(t)dt + 0.5*a(t)dt^2
        x_new = x + vx * dt + 0.5 * ax * dt**2
        y_new = y + vy * dt + 0.5 * ay * dt**2
        
        # 2. Compute New Acceleration a(t+dt)
        ax_new, ay_new = compute_coulomb_force(x_new, y_new, q1_val, m1_val)
        
        # 3. Update Velocity: v(t+dt) = v(t) + 0.5*(a(t) + a(t+dt))dt
        vx_new = vx + 0.5 * (ax + ax_new) * dt
        vy_new = vy + 0.5 * (ay + ay_new) * dt
        
        # Synchronous reassignment
        x, y = x_new, y_new
        vx, vy = vx_new, vy_new
        ax, ay = ax_new, ay_new
        
        # Store traces (first 50 particles)
        x_trace[t, :] = x[:n_traces]
        y_trace[t, :] = y[:n_traces]
        
    return x_trace, y_trace, vx, vy

# Run JIT compiled engine
print("[*] Integrating Ordinary Differential Equations (Verlet)...")
x_traces, y_traces, final_vx, final_vy = verlet_integrator(x_init, y_init, vx_init, vy_init, DT, STEPS, q1, m1)
print("[+] Simulation structurally converged and complete.")

### 3. Kinematic Visualization: Vectorial Scattering
We render the temporal matrix evolution of the tracking subset. The retrograde collision (scattering angles $> 90^{\circ}$) validates the atomic nucleus density.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Gold nucleus at the origin (0,0)
ax.plot(0, 0, marker='o', markersize=20, color='gold', label='Heavy Nucleus (Au)')

# Kinematic trajectories using Verlet history
for i in range(x_traces.shape[1]):
    ax.plot(x_traces[:, i], y_traces[:, i], linewidth=1.2, alpha=0.6, color='cyan')

ax.set_title("Rutherford Scattering Trial Integration (Discrete Verlet)", fontsize=15, pad=15)
ax.set_xlabel("Spatial Position X")
ax.set_ylabel("Impact Parameter Y / Radiant Deflection")
ax.set_xlim(-200, 200)
ax.set_ylim(-150, 150)
ax.grid(alpha=0.1)
ax.legend()

plt.show()

### 4. Analytical Consolidation: Geometric Distribution
The massive predictive power of the ensemble requires abstracting thousands of final vectors into their statistical angle $(\theta)$, to model reflection probability.

In [ ]:
# The scattering angle is measured with respect to its incoming trajectory (positive X axis).
# atan2(vy, vx) produces an angle in radians from -pi to pi.
final_angles = np.degrees(np.arctan2(final_vy, final_vx))

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(final_angles, bins=100, color='red', stat="density", log_scale=(False, True), ax=ax)

ax.set_title("Log-Statistical Distribution of Angular Deflection (5k Samples)", fontsize=14, pad=10)
ax.set_xlabel("Scattering Angle $\\theta$ (Degrees)")
ax.set_ylabel("Probability Density (Exponential Log)")

ax.axvline(0, color='yellow', linestyle='--', label='Zero frontal scattering')
# The fact that the tail exists at +/-180 confirms that certain particles "bounced" retrogradely.
plt.legend()
plt.show()